# Doublets 4 - add BCR and TCR from mixcr

Publication source copy for the Doublets 2 and 4 sequencing analyses. Outputs and execution counts are cleared. See `../VALIDATION_REPORT.md` for the separate execution results and remaining limitations.

Run from the `analysis/` directory after supplying the bundled inputs. Final figure sections explicitly load preserved manuscript checkpoints; they are distinct from fresh clustering.

In [ ]:
from pathlib import Path
Path("../results/Doublets_4_Step2_add_BCR_TCR").mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
from anndata import read_h5ad
from anndata import read_text
from collections import Counter
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from matplotlib.pyplot import rc_context
import matplotlib.colors as mcolors
import anndata
import seaborn as sns
import regex as re
import os

In [ ]:
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=200, dpi_save=300)
sc.settings.n_jobs = 30
pd.options.display.max_columns = None
pd.options.display.max_rows = None

sc.logging.print_header()

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Import unfiltered h5ad of all doublets 4 data

In [ ]:
path_data = '../data/Doublets/data/smartseq_doublets_4_analysis/'
adata = sc.read_h5ad(path_data+'Doublets4_7600cells_GeneSymbols_preprocessed_2xDoubletCounts.h5ad')

#the line below fixes an error that results when trying to run sc.tl.rank_genes_groups(adata,'leiden',method='wilcoxon')
#error is: KeyError: 'base'
adata.uns['log1p']["base"] = None
adata.obs.head()

# Filter interactions by imaging quality

In [ ]:
#filter by doublet score > 0
list_not_doubletscored = ['ArtDoub', 'Sing']
adata = adata[(adata.obs['sortedtype'].isin(list_not_doubletscored)) | (adata.obs['score']=='1')]
adata.shape

# Import TCR and BCR AIRR files

In [ ]:
list_df_tsv = []

count=0
for airr_file in os.listdir(path_data+'mixcr/exportAirr_output/'):
    if '.tsv' in airr_file:
        
        #mixcr export_Airr misses 2 columns that are needed by scirpy: cell_id, locus. Add these in manually
        df = pd.read_csv(path_data+'mixcr/exportAirr_output/'+airr_file, sep='\t')
        df['cell_id'] = '_'.join(airr_file.split('.tsv')[0].split('_')[1:])
        df['locus'] = [x[0:3] for x in df['v_call']]

        #add to master df list
        list_df_tsv.append(df)

#concatenate
df_ir = pd.concat(list_df_tsv)

#drop_duplicates within each donor, their locus, and junction. pick top one between multiple IgH
print(len(df_ir))

df_ir.drop_duplicates(subset=['cell_id','locus'],inplace=True)
print('drop duplicates',len(df_ir))

#keep only productive clones
df_ir = df_ir[(df_ir['productive']=='T')]
print('productive',len(df_ir))
df_ir = df_ir[(df_ir['complete_vdj']=='T')]
print('complete_vdj',len(df_ir))

df_ir = df_ir[(df_ir['duplicate_count']>5)]

print('duplicate_count',len(df_ir))

#reset index
df_ir.reset_index(inplace=True, drop=True)

#Extract metadata from cell name
df_ir['barcode'] = [i.split('_')[-2]+'-'+i.split('_')[-1] for i in df_ir['cell_id']]

df_ir.head()

In [ ]:
#save
df_ir.to_csv(path_data+'Doublets4_TCR_BCR_productive_completeVDJ.txt', sep='\t')
df_ir.shape

# Merge adata onto IR data

In [ ]:
merged_df = pd.merge(df_ir, adata.obs, on='barcode', how='left')
print(len(merged_df))
# Retain receptor records with matched transcriptomic population metadata.
merged_df.dropna(subset=['sortedpop'],inplace=True)

merged_df['junction_aa_count'] = [list(merged_df['junction_aa']).count(x) for x in list(merged_df['junction_aa'])]
print(len(merged_df))
merged_df.head()